# 03 — Run MILAN on the trained ResNet18
Two steps:
1. **Dissection** — compute top-k activating images for every unit in `conv1`, `layer1..layer4` (~1024 units total).
2. **Captioning** — feed those exemplars through the pretrained MILAN decoder (`base`) to get a natural-language description per unit.

In [ ]:
import os, sys, torch
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / 'milan'))
os.environ.setdefault('MILAN_DATA_DIR', str(Path.cwd().parent / 'data'))
os.environ.setdefault('MILAN_MODELS_DIR', str(Path.cwd().parent / 'models'))
os.environ.setdefault('MILAN_RESULTS_DIR', str(Path.cwd().parent / 'results'))

version_dir = Path(os.environ['MILAN_DATA_DIR']) / 'imagenet-spurious-text' / '50pct'
ckpt = Path(os.environ['MILAN_MODELS_DIR']) / 'resnet18_spurious.pth'
dissect_dir = Path(os.environ['MILAN_RESULTS_DIR']) / 'edit' / 'imagenet-spurious-text' / 'resnet18_spurious-50pct'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from milan_repro.milan_glue.run_exemplars import run as run_exemplars
run_exemplars(version_dir, ckpt, dissect_dir, device=device)

In [ ]:
from milan_repro.milan_glue.run_descriptions import run as run_descriptions
out_csv = Path(os.environ['MILAN_RESULTS_DIR']) / 'descriptions.csv'
run_descriptions(dissect_dir, out_csv, device=device)

In [ ]:
from milan_repro.editing.identify_text_neurons import annotate
annotate(out_csv, out_csv.with_name('descriptions_annotated.csv'))

In [ ]:
# Peek at a few descriptions.
import pandas as pd
df = pd.read_csv(out_csv.with_name('descriptions_annotated.csv'))
print('text neurons:', int(df['is_text_neuron'].sum()), '/', len(df))
df[df['is_text_neuron']].head(10)